In [1]:
from pathlib import Path
import os
import awkward as ak

from mltau.tools.evaluation import kinematics as k
from mltau.tools.evaluation import tagging as t
from mltau.tools.evaluation import charge_id as c
from mltau.tools.evaluation import decay_mode as d

from mltau.tools.general import reinitialize_p4
from mltau.tools.evaluation import inference
from mltau.models import SingleParTau_module


In [2]:
from hydra import compose, initialize

with initialize(version_base=None, config_path="../config", job_name="test_app"):
    cfg = compose(config_name="main")

In [3]:
OUTPUT_DIR = "/home/norman/2222_training"
RESULTS_DIR = os.path.join(OUTPUT_DIR, "results")
os.makedirs(RESULTS_DIR, exist_ok=True)

SIGNAL_SAMPLE = "z"
BKG_SAMPLE = "qq"


# SingleParTau

In [4]:
SINGLE_PARTAU_TRAININGS_DIR = "/home/norman/ml-tau/test_inference_single"

In [5]:
pred_z = ak.from_parquet("/home/norman/ml-tau/test_inference_single/kin/predictions/z_test.parquet")

In [6]:
pred_z.fields

['gen_jet_p4',
 'reco_jet_p4',
 'gen_jet_tau_p4',
 'gen_jet_tau_decaymode',
 'gen_jet_tau_charge',
 'cand_charges',
 'cand_p4',
 'tau_p4']

In [7]:
pred_z.tau_p4[0]

<Record {rho: 36.2, phi: 2.46, eta: 0.291, ...} type='Momentum4D[rho: float...'>

In [8]:
pred_z.reco_jet_p4[0]

<Record {pt: 35, eta: 0.284, phi: 2.47, ...} type='{pt: float32, eta: float...'>

In [63]:
# Tagging
sTag_sigData = ak.from_parquet(
    os.path.join(SINGLE_PARTAU_TRAININGS_DIR, "isTau", "predictions", f"{SIGNAL_SAMPLE}_test.parquet")
)
sTag_bkgData = ak.from_parquet(
    os.path.join(SINGLE_PARTAU_TRAININGS_DIR, "isTau", "predictions", f"{BKG_SAMPLE}_test.parquet")
)
sTag_evaluator = t.TaggerEvaluator(
    signal_predictions=sTag_sigData.tau_tagging_score,
    signal_gen_tau_p4=sTag_sigData.gen_jet_tau_p4,
    signal_reco_jet_p4=sTag_sigData.reco_jet_p4,
    bkg_predictions=sTag_bkgData.tau_tagging_score,
    bkg_gen_jet_p4=sTag_bkgData.gen_jet_p4,
    bkg_reco_jet_p4=sTag_bkgData.reco_jet_p4,
    cfg=cfg,
    sample=SIGNAL_SAMPLE,
    algorithm="SingleParTau",
)

In [51]:
# Decay mode
sDM_sigData = ak.from_parquet(
    os.path.join(SINGLE_PARTAU_TRAININGS_DIR, "DM", "predictions", f"{SIGNAL_SAMPLE}_test.parquet")
)
sDM_evaluator = d.DecayModeEvaluator(
    pred_proba=sDM_sigData.tau_decay_mode_probs,
    truth=sDM_sigData.gen_jet_tau_decaymode,
    output_dir=RESULTS_DIR,
    sample=SIGNAL_SAMPLE,
    algorithm="SingleParTau"
)

/opt/conda/lib/python3.11/site-packages/sklearn/metrics/_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])


In [53]:
# Charge
sCh_sigData = ak.from_parquet(
    os.path.join(SINGLE_PARTAU_TRAININGS_DIR, "charge", "predictions", f"{SIGNAL_SAMPLE}_test.parquet")
)
sCh_evaluator = c.ChargeIdEvaluator(
    predicted=sCh_sigData.tau_charge_score,
    truth=sCh_sigData.gen_jet_tau_charge,
    gen_jet_tau_p4s=sCh_sigData.gen_jet_tau_p4,
    reco_jet_p4s=sCh_sigData.reco_jet_p4,
    cfg=cfg,
    output_dir=RESULTS_DIR,
    sample=SIGNAL_SAMPLE,
    algorithm="SingleParTau",
)

ValueError: min() arg is an empty sequence

In [9]:
# Kinematics
sKin_sigData = ak.from_parquet(
    os.path.join(SINGLE_PARTAU_TRAININGS_DIR, "kin", "predictions", f"{SIGNAL_SAMPLE}_test.parquet")
)
sKin_evaluator = k.KinematicsEvaluator(
    predicted_p4=sKin_sigData.tau_p4,
    true_p4=sKin_sigData.gen_jet_tau_p4,
    cfg=cfg,
    algorithm="SingleParTau",
    sample_name=SIGNAL_SAMPLE
)

# MultiParTau

In [10]:
MULTI_PARTAU_TRAININGS_DIR = "/home/norman/ml-tau/test_inference/"
# MULTI_PARTAU_TRAININGS_DIR = "/home/laurits/0407_lifetime_training/"


In [11]:
pred_z0 = ak.from_parquet("/home/norman/ml-tau/test_inference/predictions/z_test.parquet")

In [12]:
pred_z0.fields

['gen_jet_p4',
 'reco_jet_p4',
 'gen_jet_tau_p4',
 'gen_jet_tau_decaymode',
 'gen_jet_tau_charge',
 'cand_charges',
 'cand_p4',
 'tau_tagging_score',
 'tau_charge_score',
 'tau_decay_mode',
 'tau_decay_mode_probs',
 'tau_p4']

In [13]:
multiParTau_sigData = ak.from_parquet(
    os.path.join(MULTI_PARTAU_TRAININGS_DIR, "predictions", f"{SIGNAL_SAMPLE}_test.parquet")
)
multiParTau_bkgData = ak.from_parquet(
    os.path.join(MULTI_PARTAU_TRAININGS_DIR, "predictions", f"{BKG_SAMPLE}_test.parquet")
)

mTag_evaluator = t.TaggerEvaluator(
    signal_predictions=multiParTau_sigData.tau_tagging_score,
    signal_gen_tau_p4=multiParTau_sigData.gen_jet_tau_p4,
    signal_reco_jet_p4=multiParTau_sigData.reco_jet_p4,
    bkg_predictions=multiParTau_bkgData.tau_tagging_score,
    bkg_gen_jet_p4=multiParTau_bkgData.gen_jet_p4,
    bkg_reco_jet_p4=multiParTau_bkgData.reco_jet_p4,
    cfg=cfg,
    sample=SIGNAL_SAMPLE,
    algorithm="MultiParTau",
)

mDM_evaluator = d.DecayModeEvaluator(
    pred_proba=multiParTau_sigData.tau_decay_mode_probs,
    truth=multiParTau_sigData.gen_jet_tau_decaymode,
    output_dir=RESULTS_DIR,
    sample=SIGNAL_SAMPLE,
    algorithm="MultiParTau"
)

mCh_evaluator = c.ChargeIdEvaluator(
    predicted=multiParTau_sigData.tau_charge_score,
    truth=multiParTau_sigData.gen_jet_tau_charge,
    gen_jet_tau_p4s=multiParTau_sigData.gen_jet_tau_p4,
    reco_jet_p4s=multiParTau_sigData.reco_jet_p4,
    cfg=cfg,
    output_dir=RESULTS_DIR,
    sample=SIGNAL_SAMPLE,
    algorithm="MultiParTau",
    # baseline_charges=baseline_charges,  # TODO: Add separately
)

mKin_evaluator = k.KinematicsEvaluator(
    predicted_p4=multiParTau_sigData.tau_p4,
    true_p4=multiParTau_sigData.gen_jet_tau_p4,
    cfg=cfg,
    algorithm="MultiParTau",
    sample_name=SIGNAL_SAMPLE
)

/opt/conda/lib/python3.11/site-packages/sklearn/metrics/_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])


# RecoJet

In [14]:
cand_pts = reinitialize_p4(multiParTau_sigData.cand_p4).pt
cand_charges = multiParTau_sigData.cand_charges
jet_pts = reinitialize_p4(multiParTau_sigData.reco_jet_p4).pt
baseline_charges = c.jet_charge_qkappa(
    cand_charges=cand_charges, cand_pts=cand_pts, jet_pts=jet_pts, kappa=0.5)

rCh_evaluator = c.ChargeIdEvaluator(
    predicted=baseline_charges, 
    truth=multiParTau_sigData.gen_jet_tau_charge,
    gen_jet_tau_p4s=multiParTau_sigData.gen_jet_tau_p4,
    reco_jet_p4s=multiParTau_sigData.reco_jet_p4,
    cfg=cfg,
    output_dir=RESULTS_DIR,
    sample=SIGNAL_SAMPLE,
    algorithm="RecoJet",
)

rKin_evaluator = k.KinematicsEvaluator(
    predicted_p4=multiParTau_sigData.reco_jet_p4,
    true_p4=multiParTau_sigData.gen_jet_tau_p4,
    cfg=cfg,
    algorithm="RecoJet",
    sample_name=SIGNAL_SAMPLE
)

# Combined results

In [15]:
# Kinematics
kRESULTS_DIR = os.path.join(RESULTS_DIR, "kinematics")
os.makedirs(kRESULTS_DIR, exist_ok=True)
kme = k.KinematicsMultiEvaluator(kRESULTS_DIR, cfg, sample=SIGNAL_SAMPLE)
kme.combine_results([sKin_evaluator, mKin_evaluator, rKin_evaluator])
kme.save()

# # Charge
# cRESULTS_DIR = os.path.join(RESULTS_DIR, "charge_id")
# os.makedirs(cRESULTS_DIR, exist_ok=True)
# cme = c.ChargeMultiEvaluator(cRESULTS_DIR, cfg)
# cme.combine_results([sCh_evaluator, mCh_evaluator, rCh_evaluator])
# cme.save()

# # Tagging
# tRESULTS_DIR = os.path.join(RESULTS_DIR, "tau_id")
# os.makedirs(tRESULTS_DIR, exist_ok=True)
# tme = t.TaggerMultiEvaluator(tRESULTS_DIR, cfg)
# tme.combine_results([sTag_evaluator, mTag_evaluator])
# tme.save()

# # Decay mode
# dRESULTS_DIR = os.path.join(RESULTS_DIR, "decay_mode")
# os.makedirs(dRESULTS_DIR, exist_ok=True)
# dme = d.DecayModeMultiEvaluator(dRESULTS_DIR, cfg, sample=SIGNAL_SAMPLE)
# dme.combine_results([sDM_evaluator, mDM_evaluator])
# dme.save()

/home/norman/ml-tau/ml-tau-model/mltau/tools/evaluation/kinematics.py:194: RuntimeWarning: More than 20 figures have been opened. Figures created through the pyplot interface (`matplotlib.pyplot.figure`) are retained until explicitly closed and may consume too much memory. (To control this warning, see the rcParam `figure.max_open_warning`). Consider using `matplotlib.pyplot.close()`.
  fig, rows = plt.subplots(nrows=3, ncols=4, sharex="col", figsize=(16, 9))


# Losses (maybe we want to plot some losses?)

In [ ]:
from tensorboard.backend.event_processing import event_accumulator

# log_dir = "/home/laurits/tmp/speedup_test2/tensorboard/ParTau_experiment/version_0/"

# ea = event_accumulator.EventAccumulator(log_dir)
# ea.Reload()

# # List available scalar tags
# print(ea.Tags()["scalars"])

# # Extract a specific scalar
# scalars = ea.Scalars("train_losses/decay_mode_loss")

# for s in scalars:
#     print(s.step, s.value)